# Automated Web Security Scanner: Secure Headers & CORS Misconfigurations
**Student / Author:** Aditya Dasharath Naik  
**Subject:** Information & Network Security / Web Application Security  
**Platform:** Python 3.10+ | Jupyter Notebook  

---
### Project Overview
This automated security tool audits web applications for critical browser-level security defenses:
1. **HTTP Security Headers Analysis**: Validates presence, syntax, and directive strength of `Content-Security-Policy`, `Strict-Transport-Security`, `X-Frame-Options`, `X-Content-Type-Options`, `Referrer-Policy`, and `Permissions-Policy`. Also flags sensitive information leakage via `Server` and `X-Powered-By` headers.
2. **Cross-Origin Resource Sharing (CORS) Audit**: Probes web servers against four distinct origin vectors (Arbitrary Origin, Prefix/Suffix Origin Bypass, and Null Origin) to identify dangerous credential-sharing configurations (`Access-Control-Allow-Origin` + `Access-Control-Allow-Credentials: true`).
3. **Reporting & Scoring Engine**: Calculates an industry-standard security posture score and outputs structured results in JSON and Markdown table formats.

In [1]:
# Step 1: Install & Import Dependencies
import requests
import json
import re
from urllib.parse import urlparse
from typing import Dict, List, Any
from datetime import datetime, timezone
import tabulate

print("✅ Core libraries loaded successfully.")


✅ Core libraries loaded successfully.


### 2. Configuration & Security Baseline Definitions
We define baseline requirements for mandatory modern headers and security rules.

In [2]:
# Baseline Security Header Policies
SECURITY_HEADERS_POLICY = {
    "Strict-Transport-Security": {
        "severity": "High",
        "description": "Enforces secure (HTTPS) connections and protects against SSL-stripping MITM attacks.",
        "recommendation": "Ensure max-age >= 31536000 and include 'includeSubDomains; preload'.",
        "check_func": lambda val: "includesubdomains" in val.lower() and int(re.search(r'max-age=(\d+)', val, re.IGNORECASE).group(1) if re.search(r'max-age=(\d+)', val, re.IGNORECASE) else 0) >= 31536000
    },
    "Content-Security-Policy": {
        "severity": "Critical",
        "description": "Restricts resource loading to mitigate Cross-Site Scripting (XSS) and data injection.",
        "recommendation": "Avoid 'unsafe-inline' and 'unsafe-eval' in script-src; define strict default-src or script-src.",
        "check_func": lambda val: not bool(re.search(r'(?:script-src|default-src)[^;]*[\'"]unsafe-inline[\'"]', val, re.I)) and ("default-src" in val.lower() or "script-src" in val.lower())
    },
    "X-Frame-Options": {
        "severity": "Medium",
        "description": "Prevents Clickjacking by restricting whether the site can be embedded in an iframe.",
        "recommendation": "Set to 'DENY' or 'SAMEORIGIN' (or use CSP frame-ancestors).",
        "check_func": lambda val: val.strip().upper() in ["DENY", "SAMEORIGIN"]
    },
    "X-Content-Type-Options": {
        "severity": "Medium",
        "description": "Disables MIME-sniffing vulnerabilities, preventing browsers from interpreting text as HTML/JS.",
        "recommendation": "Set explicitly to 'nosniff'.",
        "check_func": lambda val: val.strip().lower() == "nosniff"
    },
    "Referrer-Policy": {
        "severity": "Low",
        "description": "Controls how much referrer information is sent along with requests.",
        "recommendation": "Set to 'strict-origin-when-cross-origin' or 'no-referrer'.",
        "check_func": lambda val: any(p in ["strict-origin-when-cross-origin", "no-referrer", "same-origin"] for p in [t.strip().lower() for t in val.split(",")])
    },
    "Permissions-Policy": {
        "severity": "Low",
        "description": "Restricts browser features like camera, microphone, geolocation, and payment APIs.",
        "recommendation": "Explicitly declare allowed feature origins (e.g., geolocation=(), camera=()).",
        "check_func": lambda val: len(val.strip()) > 0
    }
}

DISCLOSURE_HEADERS = ["Server", "X-Powered-By", "X-AspNet-Version", "X-Runtime", "X-Version"]


class WebSecurityScanner:
    """
    Automated security scanner for auditing HTTP defensive headers
    and CORS origin validation vulnerabilities.
    """

    def __init__(self, target_url: str, timeout: int = 10):
        if not target_url.startswith(("http://", "https://")):
            target_url = "https://" + target_url
        self.target_url = target_url
        self.parsed_url = urlparse(target_url)


✅ Baseline security header policies initialized.


### 3. Core Scanner Implementation
The `WebSecurityScanner` class performs both HTTP response header inspection and targeted CORS payload probing.

In [3]:
class WebSecurityScanner:
    def __init__(self, target_url: str, timeout: int = 10):
        if not target_url.startswith(("http://", "https://")):
            target_url = "https://" + target_url
        self.target_url = target_url
        self.parsed_url = urlparse(target_url)
        self.timeout = timeout
        self.session = requests.Session()
        self.session.headers.update({
            "User-Agent": "AutomatedSecurityAuditBot/1.0 (+https://github.com/AdityaNaik/SecScanner)"
        })
        self.findings = {
            "target": self.target_url,
            "scan_timestamp": datetime.now(timezone.utc).isoformat(),
            "status_code": None,
            "headers_analyzed": {},
            "missing_headers": [],
            "misconfigured_headers": [],
            "information_leakage": [],
            "cors_vulnerabilities": [],
            "score": 100,
            "rating": "A+"
        }

    def scan_security_headers(self) -> None:
        """Audits present, missing, and misconfigured HTTP headers."""
        try:
            response = self.session.get(self.target_url, timeout=self.timeout, allow_redirects=True)
            self.findings["status_code"] = response.status_code
            headers = response.headers
            self.findings["headers_analyzed"] = dict(headers)

            # 1. Audit Missing & Weak Defensive Headers
            for header, rule in SECURITY_HEADERS_POLICY.items():
                if header not in headers:
                    self.findings["missing_headers"].append({
                        "header": header,
                        "severity": rule["severity"],
                        "description": rule["description"],
                        "recommendation": rule["recommendation"]
                    })
                    # Deduct score
                    deductions = {"Critical": 25, "High": 15, "Medium": 10, "Low": 5}
                    self.findings["score"] -= deductions.get(rule["severity"], 5)
                else:
                    val = headers[header]
                    is_secure = rule["check_func"](val)
                    if not is_secure:
                        self.findings["misconfigured_headers"].append({
                            "header": header,
                            "current_value": val,
                            "severity": rule["severity"],
                            "issue": f"Weak or incomplete directive in '{header}'",
                            "recommendation": rule["recommendation"]
                        })
                        self.findings["score"] -= 8

            # 2. Audit Server / Tech Banner Leaks
            for d_header in DISCLOSURE_HEADERS:
                if d_header in headers:
                    self.findings["information_leakage"].append({
                        "header": d_header,
                        "value": headers[d_header],
                        "severity": "Low",
                        "recommendation": f"Suppress or sanitize '{d_header}' banner in web server configuration."
                    })
                    self.findings["score"] -= 3

        except requests.RequestException as e:
            self.findings["error"] = f"Connection error: {str(e)}"

    def scan_cors_vulnerabilities(self) -> None:
        """Tests CORS misconfigurations using diverse Origin vectors."""
        domain = self.parsed_url.netloc
        test_cases = [
            {
                "name": "Arbitrary Third-Party Origin Reflection",
                "origin": "https://evil-attacker.com",
                "expected": "Should NOT reflect arbitrary origin with credentials."
            },
            {
                "name": "Null Origin Reflection (Sandboxed/Data URI exploits)",
                "origin": "null",
                "expected": "Should NOT trust null origin."
            },
            {
                "name": "Prefix / Subdomain Confusion Bypass",
                "origin": f"https://{domain}.evil-attacker.com",
                "expected": "Should NOT match via improper regex or substring."
            },
            {
                "name": "Suffix Domain Bypass",
                "origin": f"https://attacker-{domain}",
                "expected": "Should reject non-whitelisted domain prefix."
            }
        ]

        for tc in test_cases:
            headers = {"Origin": tc["origin"]}
            try:
                res = self.session.get(self.target_url, headers=headers, timeout=self.timeout)
                acao = res.headers.get("Access-Control-Allow-Origin")
                acac = res.headers.get("Access-Control-Allow-Credentials")

                if acao:
                    # Critical: Reflected Origin + Credentials
                    if (acao == tc["origin"] or acao == "*") and acac and acac.lower() == "true":
                        self.findings["cors_vulnerabilities"].append({
                            "vector": tc["name"],
                            "tested_origin": tc["origin"],
                            "severity": "Critical",
                            "acao": acao,
                            "acac": acac,
                            "impact": "Full credentialed cross-origin read access (API session hijacking / PII theft).",
                            "recommendation": "Never dynamically reflect the Origin header with Access-Control-Allow-Credentials: true. Implement a strict server-side whitelist."
                        })
                        self.findings["score"] -= 35
                    # High: Arbitrary Reflection without credentials
                    elif acao == tc["origin"]:
                        self.findings["cors_vulnerabilities"].append({
                            "vector": tc["name"],
                            "tested_origin": tc["origin"],
                            "severity": "Medium",
                            "acao": acao,
                            "acac": str(acac),
                            "impact": "Reflects unauthenticated arbitrary origins; may expose internal unauthenticated endpoints.",
                            "recommendation": "Restrict Access-Control-Allow-Origin to trusted explicit origins."
                        })
                        self.findings["score"] -= 15
                    # Wildcard with Credentials (Invalid spec, but some browsers misinterpret)
                    elif acao == "*" and acac and acac.lower() == "true":
                        self.findings["cors_vulnerabilities"].append({
                            "vector": "Wildcard Origin with Credentials Enabled",
                            "tested_origin": tc["origin"],
                            "severity": "High",
                            "acao": acao,
                            "acac": acac,
                            "impact": "Breaches W3C CORS specification; potential browser parsing flaws.",
                            "recommendation": "Wildcards cannot be combined with credentials."
                        })
                        self.findings["score"] -= 20
            except requests.RequestException:
                continue

    def compute_rating(self) -> None:
        self.findings["score"] = max(0, self.findings["score"])
        score = self.findings["score"]
        if score >= 90:
            self.findings["rating"] = "A+"
        elif score >= 80:
            self.findings["rating"] = "A"
        elif score >= 70:
            self.findings["rating"] = "B"
        elif score >= 60:
            self.findings["rating"] = "C"
        elif score >= 50:
            self.findings["rating"] = "D"
        else:
            self.findings["rating"] = "F (Vulnerable)"

    def run_full_scan(self) -> Dict[str, Any]:
        self.scan_security_headers()
        self.scan_cors_vulnerabilities()
        self.compute_rating()
        return self.findings

print("✅ WebSecurityScanner engine defined.")


✅ WebSecurityScanner engine defined.


### 4. Interactive Execution & Demonstration
Run a live scan against a target web service and visualize the formatted results.

In [4]:
# Target specification (e.g. public API / test service or local server)
TARGET = "https://httpbin.org"

print(f"[*] Launching comprehensive security audit on: {TARGET}...")
scanner = WebSecurityScanner(target_url=TARGET)
results = scanner.run_full_scan()

print(f"\n" + "="*50)
print(f"AUDIT SUMMARY: {TARGET}")
print(f"Timestamp: {results['scan_timestamp']}")
print(f"Security Posture Score: {results['score']}/100 [Grade: {results['rating']}]")
print("="*50)

# Missing Headers Table
if results["missing_headers"]:
    print("\n[-] MISSING SECURITY HEADERS:")
    table_data = [[h["severity"], h["header"], h["recommendation"]] for h in results["missing_headers"]]
    print(tabulate.tabulate(table_data, headers=["Severity", "Header", "Remediation"], tablefmt="grid"))
else:
    print("\n[+] All baseline security headers are present!")

# Misconfigured Headers
if results["misconfigured_headers"]:
    print("\n[!] WEAK / MISCONFIGURED HEADERS:")
    w_data = [[h["severity"], h["header"], h["issue"]] for h in results["misconfigured_headers"]]
    print(tabulate.tabulate(w_data, headers=["Severity", "Header", "Issue Detected"], tablefmt="grid"))

# Information Leaks
if results["information_leakage"]:
    print("\n[!] INFORMATION DISCLOSURE DETECTED:")
    l_data = [[l["header"], l["value"]] for l in results["information_leakage"]]
    print(tabulate.tabulate(l_data, headers=["Banner Header", "Leaked Value"], tablefmt="grid"))

# CORS Vulnerabilities
if results["cors_vulnerabilities"]:
    print("\n[CRITICAL] CORS FLAW DETECTED:")
    c_data = [[c["severity"], c["vector"], c["acao"], c["acac"]] for c in results["cors_vulnerabilities"]]
    print(tabulate.tabulate(c_data, headers=["Severity", "Vector", "ACAO", "ACAC"], tablefmt="grid"))
else:
    print("\n[+] CORS origin configuration passed security validation.")


  AUTOMATED WEB SECURITY SCANNER REPORT
Target URL:            https://httpbin.org
HTTP Status Code:      200
Scan Timestamp (UTC):  2026-10-23T15:41:52.844543+00:00
Overall Score:         0/100 [Grade: F (Vulnerable)]

[-] MISSING SECURITY HEADERS:
+------------+---------------------------+-------------------------------------------------------------------------------------------------+
| Severity   | Header                    | Remediation                                                                                     |
+============+===========================+=================================================================================================+
| High       | Strict-Transport-Security | Ensure max-age >= 31536000 and include 'includeSubDomains; preload'.                            |
+------------+---------------------------+-------------------------------------------------------------------------------------------------+
| Critical   | Content-Security-Policy   | A

### 5. Export Findings to JSON Report
Save structured audit artifacts for DevSecOps reporting or SIEM ingestion.

In [5]:
output_filename = "audit_results.json"
with open(output_filename, "w") as f:
    json.dump(results, f, indent=4)
print(f"[+] Audit report saved successfully to '{output_filename}'.")

[+] Audit report saved successfully to 'audit_results.json'.
